# 🧱 Part 11: Data Engineering: Cleaning the Fuel for Training

> **Previous context**: Scaling laws showed data volume matters. This part asks what makes data usable.
> **Goal for this part**: Understand extraction, language filtering, quality filtering, deduplication, and data mixing.

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. Why data engineering matters

A model learns patterns from its corpus. Messy, duplicated, or low-quality text teaches messy behavior.

## 1. Pipeline steps

A typical pipeline extracts text, filters languages, removes low-quality documents, deduplicates near-copies, and mixes sources deliberately.

## 2. Quality filters

Simple rules can catch extreme length, symbol noise, repeated text, or strange word statistics. Model-based scores can add another signal.

## 3. Deduplication

Exact hashing catches identical documents; MinHash-style methods catch near duplicates.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- [ ] Data quality affects model behavior.
- [ ] Filtering and deduplication are core training infrastructure.
- [ ] Data mixing controls what the model sees often.

Next, continue through the code cells for the Training Systems part and inspect the printed observations.


In [ ]:
import re
import hashlib
import numpy as np

np.random.seed(42)

In [ ]:
# Teaching note: follow this line to see the main step.
raw_html = """
<!DOCTYPE html>
<html>
<head>
  <title>What is Machine Learning? - AI Blog</title>
  <meta name="description" content="Learn ML basics">
  <script src="tracking.js"></script>
  <style>.ad { color: red; }</style>
</head>
<body>
  <nav>
    <a href="/">Home</a> |
    <a href="/about">About</a> |
    <a href="/contact">Contact</a>
  </nav>
  
  <div class="sidebar">
    <div class="ad">
      <h3>Sponsored</h3>
      <p>Buy the best AI course! 50% off today only!</p>
      <button>Click Here!</button>
    </div>
    <script type="text/javascript">
      var _gaq = _gaq || [];
      _gaq.push(['_setAccount', 'UA-12345-6']);
      _gaq.push(['_trackPageview']);
    </script>
  </div>
  
  <article>
    <h1>What is Machine Learning?</h1>
    <p>Machine learning is a subset of artificial intelligence
    that enables systems to learn and improve from experience
    without being explicitly programmed.</p>
    
    <p>The process of learning begins with observations or data,
    such as examples, direct experience, or instruction.</p>
    
    <p>Machine learning algorithms build a mathematical model
    based on sample data, known as "training data".</p>
  </article>
  
  <div class="comments">
    <p>User123: Great article!</p>
    <p>Bot456: Buy followers at cheap-followers.com!</p>
  </div>
  
  <footer>
    <p>Copyright 2024 AI Blog. All rights reserved.</p>
    <p>Terms of Service | Privacy Policy | Cookie Settings</p>
  </footer>
</body>
</html>
"""

print('Read the values printed above and connect them to the concept in this cell.')
print(raw_html[:500])
print("...")
print()
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

# Teaching note: follow this line to see the main step.
def remove_scripts_styles(html):
    'Read the values printed above and connect them to the concept in this cell.'
    html = re.sub(r'<script[^>]*>.*?</script>', '', html, flags=re.DOTALL | re.IGNORECASE)
    html = re.sub(r'<style[^>]*>.*?</style>', '', html, flags=re.DOTALL | re.IGNORECASE)
    return html

# Teaching note: follow this line to see the main step.
def strip_tags(html):
    'Read the values printed above and connect them to the concept in this cell.'
    return re.sub(r'<[^>]+>', '', html)

# Teaching note: follow this line to see the main step.
def clean_whitespace(text):
    'Read the values printed above and connect them to the concept in this cell.'
    text = re.sub(r'[ \t]+', ' ', text)  # Teaching note: follow this line to see the main step.
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)  # Teaching note: follow this line to see the main step.
    return text.strip()

# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
no_scripts = remove_scripts_styles(raw_html)
print('Read the values printed above and connect them to the concept in this cell.')
print()

print('Read the values printed above and connect them to the concept in this cell.')
plain_text = strip_tags(no_scripts)
print('Read the values printed above and connect them to the concept in this cell.')
print()

print('Read the values printed above and connect them to the concept in this cell.')
clean_text = clean_whitespace(plain_text)
print('Read the values printed above and connect them to the concept in this cell.')
print()

print("=" * 60)
print('Read the values printed above and connect them to the concept in this cell.')
print("=" * 60)
print(clean_text)
print("=" * 60)
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

# Teaching note: follow this line to see the main step.
samples = [
    {"name": 'Read the values printed above and connect them to the concept in this cell.', "text": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. The field has grown rapidly since the 2010s, driven by advances in deep learning and the availability of large datasets."},
    {"name": 'Read the values printed above and connect them to the concept in this cell.', "text": "BUY NOW!!! Click here!!! Limited time offer!!! Subscribe today and get 50% OFF!!! Don't miss this opportunity!!!"},
    {"name": 'Read the values printed above and connect them to the concept in this cell.', "text": "Chapter 1. Introduction. Chapter 2. Methods. Chapter 3. Results. Chapter 4. Discussion. Chapter 5. Conclusion. Appendix A. Appendix B. References."},
    {"name": 'Read the values printed above and connect them to the concept in this cell.', "text": "Hello world."},
    {"name": 'Read the values printed above and connect them to the concept in this cell.', "text": "asdfjkl; qwerty zxcvbnm @#$%@# 123456789 !!!!!!!"},
    {"name": 'Read the values printed above and connect them to the concept in this cell.', "text": "This is a blog post.\n" * 30 + "unique content here"},
]

def quality_check(text):
    'Reason'
    
    # Teaching note: follow this line to see the main step.
    words = text.split()
    if len(words) < 5:
        return False, 'Read the values printed above and connect them to the concept in this cell.'
    if len(text) > 5000:
        return False, 'Read the values printed above and connect them to the concept in this cell.'
    
    # Teaching note: follow this line to see the main step.
    avg_word_len = np.mean([len(w) for w in words])
    if avg_word_len > 12:
        return False, 'Read the values printed above and connect them to the concept in this cell.'
    
    # Teaching note: follow this line to see the main step.
    special_count = sum(1 for c in text if not c.isalnum() and not c.isspace())
    special_ratio = special_count / max(len(text), 1)
    if special_ratio > 0.25:
        return False, 'Read the values printed above and connect them to the concept in this cell.'
    
    # Teaching note: follow this line to see the main step.
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) > 3:
        unique_ratio = len(set(lines)) / len(lines)
        if unique_ratio < 0.4:
            return False, 'Read the values printed above and connect them to the concept in this cell.'
    
    # Teaching note: follow this line to see the main step.
    if len(text) > 50:
        upper_ratio = sum(1 for c in text if c.isupper()) / sum(1 for c in text if c.isalpha())
        if upper_ratio > 0.5:
            return False, 'Read the values printed above and connect them to the concept in this cell.'
    
    return True, 'Exercise passed: you have understood this step.'


print('Reason')
print("-" * 55)
for sample in samples:
    passed, reason = quality_check(sample['text'])
    status = 'Read the values printed above and connect them to the concept in this cell.' if passed else 'Read the values printed above and connect them to the concept in this cell.'
    print(f"{sample['name']:<12s} {status:>8s}  {reason}")

print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

# Teaching note: follow this line to see the main step.
docs = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "Machine learning is a subset of artificial intelligence.",  # Teaching note: follow this line to see the main step.
    "Natural language processing deals with text data.",
    "Deep learning uses neural networks with many layers.",  # Teaching note: follow this line to see the main step.
]

print('Read the values printed above and connect them to the concept in this cell.')
print()

# Teaching note: follow this line to see the main step.
seen = set()
unique_docs = []

for i, doc in enumerate(docs):
    # Teaching note: follow this line to see the main step.
    fingerprint = hashlib.sha256(doc.encode()).hexdigest()[:16]  # Teaching note: follow this line to see the main step.
    is_new = fingerprint not in seen
    
    print('Read the values printed above and connect them to the concept in this cell.')
    
    if is_new:
        seen.add(fingerprint)
        unique_docs.append(doc)

print()
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print()

# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

# Teaching note: follow this line to see the main step.
doc_A = "the cat sat on the mat and looked at the dog"
doc_B = "the cat sat on the mat and watched the dog"  # Teaching note: follow this line to see the main step.
doc_C = "quantum mechanics describes behavior of subatomic particles"  # Teaching note: follow this line to see the main step.

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()

# Teaching note: follow this line to see the main step.
def get_ngrams(text, n=3):
    words = text.lower().split()
    return set(' '.join(words[i:i+n]) for i in range(len(words) - n + 1))

A_ngrams = get_ngrams(doc_A, 3)
B_ngrams = get_ngrams(doc_B, 3)
C_ngrams = get_ngrams(doc_C, 3)

print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print()

# Teaching note: follow this line to see the main step.
def jaccard(s1, s2):
    inter = len(s1 & s2)
    union = len(s1 | s2)
    return inter / union if union > 0 else 0

j_AB = jaccard(A_ngrams, B_ngrams)
j_AC = jaccard(A_ngrams, C_ngrams)

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print(f"Jaccard(A, B) = {len(A_ngrams & B_ngrams)}/{len(A_ngrams | B_ngrams)} = {j_AB:.2%}")
print()
print(f"Jaccard(A, C) = {j_AC:.2%}")
print()

# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

# Teaching note: follow this line to see the main step.
def minhash_signature(ngrams, num_hashes=4):
    'Read the values printed above and connect them to the concept in this cell.'
    sig = []
    for i in range(num_hashes):
        # Teaching note: follow this line to see the main step.
        min_val = float('inf')
        for ng in ngrams:
            h = hash(ng + str(i)) % 100000
            min_val = min(min_val, h)
        sig.append(min_val) if min_val != float('inf') else sig.append(0)
    return sig

sig_A = minhash_signature(A_ngrams)
sig_B = minhash_signature(B_ngrams)
sig_C = minhash_signature(C_ngrams)

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()

# Teaching note: follow this line to see the main step.
def minhash_sim(s1, s2):
    matches = sum(1 for a, b in zip(s1, s2) if a == b)
    return matches / len(s1)

mh_AB = minhash_sim(sig_A, sig_B)
mh_AC = minhash_sim(sig_A, sig_C)

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

sources = [
    ('Read the values printed above and connect them to the concept in this cell.', 10000, 0.6, 1),
    ("Wikipedia",               100, 0.95, 4),
    ("Books",                   500, 0.85, 2),
    ("Code (GitHub)",          1000, 0.75, 2),
    ("ArXiv Papers",             50, 0.9,  4),
    ("News",                    300, 0.7,  1),
]

print('Read the values printed above and connect them to the concept in this cell.')
print("-" * 72)

total_effective = 0
results = []
for name, size, quality, epochs in sources:
    effective = size * epochs
    total_effective += effective
    results.append((name, size, quality, epochs, effective))

for name, size, quality, epochs, effective in results:
    ratio = effective / total_effective * 100
    print(f"{name:<25s} {size:>6.0f}B   {quality:>5.0%}  {epochs:>4d}x  {effective:>8.0f}B   {ratio:>6.1f}%")

print()
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Training')

In [ ]:
print('Read the values printed above and connect them to the concept in this cell.')
print()

steps = [
    ('Read the values printed above and connect them to the concept in this cell.', [
        'Read the values printed above and connect them to the concept in this cell.',
        'Training',
        'Read the values printed above and connect them to the concept in this cell.',
    ]),
    ('Read the values printed above and connect them to the concept in this cell.', [
        'Read the values printed above and connect them to the concept in this cell.',
        'Read the values printed above and connect them to the concept in this cell.',
    ]),
    ('Read the values printed above and connect them to the concept in this cell.', [
        'Read the values printed above and connect them to the concept in this cell.',
        'Read the values printed above and connect them to the concept in this cell.',
        'Read the values printed above and connect them to the concept in this cell.',
    ]),
    ('Read the values printed above and connect them to the concept in this cell.', [
        'Read the values printed above and connect them to the concept in this cell.',
        'Read the values printed above and connect them to the concept in this cell.',
        'Read the values printed above and connect them to the concept in this cell.',
    ]),
    ('Read the values printed above and connect them to the concept in this cell.', [
        'Read the values printed above and connect them to the concept in this cell.',
        'Read the values printed above and connect them to the concept in this cell.',
        'Read the values printed above and connect them to the concept in this cell.',
    ]),
    ('Read the values printed above and connect them to the concept in this cell.', [
        "Wikipedia (2x epoch): 4B tokens",
        "Books (2x epoch): 6B tokens",
        "Code GitHub (2x epoch): 10B tokens",
        'Read the values printed above and connect them to the concept in this cell.',
        'Read the values printed above and connect them to the concept in this cell.',
    ]),
    ('Read the values printed above and connect them to the concept in this cell.', [
        'Read the values printed above and connect them to the concept in this cell.',
        'Read the values printed above and connect them to the concept in this cell.',
        'Read the values printed above and connect them to the concept in this cell.',
        'Training',
    ]),
]

for title, details in steps:
    print(title)
    for d in details:
        print(f"  {d}")
    print()

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')